# NanoGPT with TurboQuant

**nanoGPT + TurboQuant KV Cache Compression**

This notebook demonstrates:
1. TurboQuant algorithm internals (Lloyd-Max codebook, random rotation, QJL)
2. KV cache compression quality benchmarks
3. Training a small GPT on Shakespeare
4. Inference comparison: Standard vs TurboQuant

**References:**
- [nanoGPT](https://github.com/karpathy/nanoGPT) - Andrey Karpathy
- [TurboQuant](https://arxiv.org/abs/2504.19874) - Zandieh, Daliri, Hadian, Mirrokni (ICLR 2026)

## 0. Setup

In [ ]:
import os, sys

# Clone the repository
REPO_DIR = 'NanoGPT_with_TurboQuant'

if not os.path.exists(REPO_DIR):
    !git clone https://github.com/3t14/NanoGPT_with_TurboQuant.git

os.chdir(REPO_DIR)
sys.path.insert(0, os.getcwd())
print(f'Working directory: {os.getcwd()}')

In [ ]:
import torch
import numpy as np
import math
import time
import matplotlib.pyplot as plt

device = 'cuda' if torch.cuda.is_available() else 'cpu'
print(f'Device: {device}')
print(f'PyTorch: {torch.__version__}')
if device == 'cuda':
    print(f'GPU: {torch.cuda.get_device_name()}')
    print(f'Memory: {torch.cuda.mem_get_info()[1] / 1e9:.1f} GB')

## 1. TurboQuant Algorithm Deep Dive

TurboQuant compresses KV cache vectors using two stages:

**Stage 1 (PolarQuant):** Random orthogonal rotation transforms each vector so coordinates follow N(0, 1/d). A precomputed Lloyd-Max quantizer optimally quantizes each coordinate.

**Stage 2 (QJL):** Projects the quantization residual through a random matrix and stores only sign bits. This corrects inner product bias.

### 1.1 Lloyd-Max Optimal Codebook

After random rotation, each coordinate of a unit vector follows approximately N(0, 1/d). The Lloyd-Max algorithm finds the optimal scalar quantizer (minimizing MSE) for this known distribution.

In [ ]:
from turboquant import solve_lloyd_max, get_codebook

fig, axes = plt.subplots(1, 3, figsize=(15, 4))

for idx, bits in enumerate([2, 3, 4]):
    ax = axes[idx]
    d = 64
    centroids, boundaries = get_codebook(d, bits)
    sigma = 1.0 / math.sqrt(d)
    n_levels = 2 ** bits

    # Plot the Gaussian PDF
    x = np.linspace(-4 * sigma, 4 * sigma, 500)
    pdf = (1 / np.sqrt(2 * np.pi * sigma**2)) * np.exp(-x**2 / (2 * sigma**2))
    ax.plot(x, pdf, 'b-', linewidth=2, label='N(0, 1/d)')

    # Plot centroids
    for c in centroids:
        ax.axvline(c.item(), color='red', linestyle='--', alpha=0.7, linewidth=1.5)

    # Plot boundaries
    for b in boundaries:
        ax.axvline(b.item(), color='green', linestyle=':', alpha=0.5)

    ax.set_title(f'{bits}-bit ({n_levels} levels)', fontsize=14)
    ax.set_xlabel('Coordinate value')
    if idx == 0:
        ax.set_ylabel('Probability density')
    ax.legend(['PDF', 'Centroids', 'Boundaries'], fontsize=9)

plt.suptitle(f'Lloyd-Max Optimal Quantizer for d={d} (sigma={sigma:.4f})', fontsize=14, y=1.02)
plt.tight_layout()
plt.show()

# Print codebook values
for bits in [2, 3, 4]:
    c, b = get_codebook(64, bits)
    print(f'{bits}-bit centroids: {[f"{v:.4f}" for v in c.tolist()]}')

### 1.2 Random Rotation Effect

After multiplying by a random orthogonal matrix, the coordinates of any vector become approximately i.i.d. Gaussian.

In [ ]:
from turboquant import generate_rotation_matrix

d = 64
Pi = generate_rotation_matrix(d, seed=42)

# Generate structured (non-Gaussian) vectors
n_vecs = 5000
x = torch.randn(n_vecs, d)
# Make them sparse: zero out 80% of coordinates
mask = torch.rand(n_vecs, d) > 0.8
x = x * mask.float()
x = x / (torch.norm(x, dim=-1, keepdim=True) + 1e-8)  # normalize

# Rotate
x_rotated = x @ Pi.T

fig, axes = plt.subplots(1, 2, figsize=(12, 4))

# Before rotation
axes[0].hist(x[:, 0].numpy(), bins=80, density=True, alpha=0.7, color='steelblue')
axes[0].set_title('Before rotation (coord 0)', fontsize=13)
axes[0].set_xlabel('Value')
axes[0].set_ylabel('Density')

# After rotation
sigma = 1.0 / math.sqrt(d)
t = np.linspace(-4*sigma, 4*sigma, 200)
gaussian = (1 / np.sqrt(2*np.pi*sigma**2)) * np.exp(-t**2 / (2*sigma**2))

axes[1].hist(x_rotated[:, 0].numpy(), bins=80, density=True, alpha=0.7, color='coral', label='Empirical')
axes[1].plot(t, gaussian, 'k-', linewidth=2, label=f'N(0, 1/{d})')
axes[1].set_title('After rotation (coord 0)', fontsize=13)
axes[1].set_xlabel('Value')
axes[1].legend()

plt.suptitle('Random Rotation Makes Coordinates Gaussian', fontsize=14, y=1.02)
plt.tight_layout()
plt.show()

print(f'Before rotation - mean: {x[:, 0].mean():.4f}, std: {x[:, 0].std():.4f}')
print(f'After rotation  - mean: {x_rotated[:, 0].mean():.4f}, std: {x_rotated[:, 0].std():.4f}')
print(f'Expected std (1/sqrt(d)): {sigma:.4f}')

### 1.3 TurboQuant Compression Quality

Test how well TurboQuant preserves attention scores at different bit-widths.

In [ ]:
from turboquant import TurboQuantKVCompressor

B, H, T, D = 2, 8, 128, 64
torch.manual_seed(42)

results = {}
for bits in [2, 3, 4]:
    comp = TurboQuantKVCompressor(D, bits=bits, device='cpu')

    k = torch.randn(B, H, T, D)
    v = torch.randn(B, H, T, D)
    q = torch.randn(B, H, T, D)

    # Compress
    ck = comp.compress_keys(k)
    cv = comp.compress_values(v)
    v_hat = comp.decompress_values(cv)

    # True vs TurboQuant attention scores
    scale = 1.0 / math.sqrt(D)
    true_scores = torch.matmul(q.float(), k.float().transpose(-2, -1)) * scale
    tq_scores = comp.attention_scores(q, ck)

    cos_sim = torch.nn.functional.cosine_similarity(
        true_scores.reshape(-1).unsqueeze(0),
        tq_scores.reshape(-1).unsqueeze(0)
    ).item()
    score_mse = ((true_scores - tq_scores)**2).mean().item()
    v_mse = ((v - v_hat)**2).mean().item()
    info = comp.memory_savings_info(T, H)

    results[bits] = {
        'cos_sim': cos_sim,
        'score_mse': score_mse,
        'value_mse': v_mse,
        'ratio': info['compression_ratio'],
        'bits_per_entry': info['bits_per_entry'],
    }

    print(f'{bits}-bit TurboQuant:')
    print(f'  Attention cosine similarity: {cos_sim:.4f}')
    print(f'  Attention score MSE:         {score_mse:.6f}')
    print(f'  Value reconstruction MSE:    {v_mse:.6f}')
    print(f'  Compression ratio:           {info["compression_ratio"]:.2f}x')
    print(f'  Effective bits/entry:         {info["bits_per_entry"]:.2f}')
    print()

In [ ]:
# Visualize attention score comparison
comp_3bit = TurboQuantKVCompressor(D, bits=3, device='cpu')
torch.manual_seed(42)
k = torch.randn(1, 1, 64, D)
q = torch.randn(1, 1, 64, D)

ck = comp_3bit.compress_keys(k)
scale = 1.0 / math.sqrt(D)
true_scores = (torch.matmul(q.float(), k.float().transpose(-2, -1)) * scale)[0, 0]
tq_scores = comp_3bit.attention_scores(q, ck)[0, 0]

fig, axes = plt.subplots(1, 3, figsize=(16, 4))

vmin = min(true_scores.min().item(), tq_scores.min().item())
vmax = max(true_scores.max().item(), tq_scores.max().item())

im0 = axes[0].imshow(true_scores.detach().numpy(), cmap='RdBu_r', vmin=vmin, vmax=vmax, aspect='auto')
axes[0].set_title('Standard Attention Scores', fontsize=12)
axes[0].set_xlabel('Key position')
axes[0].set_ylabel('Query position')

im1 = axes[1].imshow(tq_scores.detach().numpy(), cmap='RdBu_r', vmin=vmin, vmax=vmax, aspect='auto')
axes[1].set_title('TurboQuant 3-bit Scores', fontsize=12)
axes[1].set_xlabel('Key position')

diff = (true_scores - tq_scores).detach().numpy()
im2 = axes[2].imshow(diff, cmap='RdBu_r', aspect='auto')
axes[2].set_title(f'Difference (MSE={np.mean(diff**2):.4f})', fontsize=12)
axes[2].set_xlabel('Key position')

for ax, im in zip(axes, [im0, im1, im2]):
    plt.colorbar(im, ax=ax, fraction=0.046)

plt.tight_layout()
plt.show()

### 1.4 Compression Ratio Summary

In [ ]:
bits_list = [2, 3, 4]
ratios = [results[b]['ratio'] for b in bits_list]
cos_sims = [results[b]['cos_sim'] for b in bits_list]

fig, ax1 = plt.subplots(figsize=(8, 5))
ax2 = ax1.twinx()

bars = ax1.bar([str(b) for b in bits_list], ratios, color='steelblue', alpha=0.7, label='Compression Ratio')
line = ax2.plot([str(b) for b in bits_list], cos_sims, 'ro-', linewidth=2, markersize=10, label='Cosine Similarity')

ax1.set_xlabel('Quantization Bits', fontsize=13)
ax1.set_ylabel('Compression Ratio (x)', fontsize=13, color='steelblue')
ax2.set_ylabel('Attention Score Cosine Similarity', fontsize=13, color='red')
ax2.set_ylim(0.8, 1.0)

# Add value labels
for bar, r in zip(bars, ratios):
    ax1.text(bar.get_x() + bar.get_width()/2., bar.get_height() + 0.1,
             f'{r:.1f}x', ha='center', fontsize=12, color='steelblue', fontweight='bold')
for i, (b, cs) in enumerate(zip(bits_list, cos_sims)):
    ax2.text(i, cs + 0.005, f'{cs:.3f}', ha='center', fontsize=11, color='red', fontweight='bold')

plt.title('TurboQuant: Compression vs Quality Tradeoff', fontsize=14)
fig.legend(loc='upper right', bbox_to_anchor=(0.88, 0.88), fontsize=11)
plt.tight_layout()
plt.show()

## 2. Train a Shakespeare Model

Train a small character-level GPT on Tiny Shakespeare (~1M characters).

In [ ]:
# Prepare the dataset
!pip install requests numpy -q
!python data/shakespeare_char/prepare.py

In [ ]:
# Train a small model (~5 min on T4 GPU)
# Reduce max_iters for a quicker demo if needed
!python train.py config/train_shakespeare_char.py \
    --device={device} \
    --compile=False \
    --max_iters=2000 \
    --eval_interval=200 \
    --use_turboquant=True \
    --turboquant_bits=3

## 3. Generate Text: Standard vs TurboQuant

Compare text generation with and without TurboQuant KV cache compression.

In [ ]:
import pickle
from model import GPT, GPTConfig

# Load trained model
out_dir = 'out-shakespeare-char'
ckpt = torch.load(f'{out_dir}/ckpt.pt', map_location=device)

model_args = ckpt['model_args']
model_args['use_turboquant'] = True
model_args['turboquant_bits'] = 3

config = GPTConfig(**model_args)
model = GPT(config)

state_dict = ckpt['model']
for k in list(state_dict.keys()):
    if k.startswith('_orig_mod.'):
        state_dict[k[len('_orig_mod.'):]] = state_dict.pop(k)
model.load_state_dict(state_dict)
model.eval()
model.to(device)

# Load tokenizer
with open('data/shakespeare_char/meta.pkl', 'rb') as f:
    meta = pickle.load(f)
stoi, itos = meta['stoi'], meta['itos']
encode = lambda s: [stoi[c] for c in s]
decode = lambda l: ''.join([itos[i] for i in l])

print(f'Model loaded: {model.get_num_params()/1e6:.2f}M parameters')
print(f'Config: {config.n_layer} layers, {config.n_head} heads, {config.n_embd} embd')

In [ ]:
def generate_text(prompt, max_tokens=300, use_turboquant=False, temperature=0.8, top_k=100):
    """Generate text and measure speed."""
    start_ids = encode(prompt)
    x = torch.tensor(start_ids, dtype=torch.long, device=device)[None, ...]

    torch.manual_seed(42)
    t0 = time.time()
    with torch.no_grad():
        y = model.generate(x, max_tokens, temperature=temperature, top_k=top_k,
                           use_turboquant=use_turboquant)
    elapsed = time.time() - t0
    text = decode(y[0].tolist())
    tok_per_sec = max_tokens / elapsed
    return text, elapsed, tok_per_sec

prompt = "ROMEO:\nO, she doth teach the torches to burn bright!\n"
max_tokens = 300

print('=' * 60)
print('STANDARD (Full Precision Attention)')
print('=' * 60)
text_std, t_std, tps_std = generate_text(prompt, max_tokens, use_turboquant=False)
print(text_std)
print(f'\nTime: {t_std:.2f}s | {tps_std:.1f} tok/s')

print('\n' + '=' * 60)
print('TURBOQUANT 3-bit (Compressed KV Cache)')
print('=' * 60)
text_tq, t_tq, tps_tq = generate_text(prompt, max_tokens, use_turboquant=True)
print(text_tq)
print(f'\nTime: {t_tq:.2f}s | {tps_tq:.1f} tok/s')

In [ ]:
# Memory savings estimate
from turboquant import TurboQuantKVCompressor

head_dim = config.n_embd // config.n_head
for bits in [2, 3, 4]:
    comp = TurboQuantKVCompressor(head_dim, bits)
    info = comp.memory_savings_info(seq_len=max_tokens, n_heads=config.n_head)
    print(f'{bits}-bit: {info["compression_ratio"]:.1f}x compression '
          f'({info["fp16_bits"]/8/1024:.1f} KB -> {info["compressed_bits"]/8/1024:.1f} KB per layer)')

## 4. Quality Analysis at Different Bit-widths

Generate with different TurboQuant bit-widths and compare output quality.

In [ ]:
prompt = "HAMLET:\nTo be, or not to be, that is the question:\n"

for bits in [None, 4, 3, 2]:
    label = f'TQ-{bits}bit' if bits else 'Standard'
    use_tq = bits is not None

    # Temporarily change bits if needed
    if use_tq:
        for block in model.transformer.h:
            block.attn.turboquant_bits = bits
            block.attn._tq_compressor = None  # force re-init

    text, elapsed, tps = generate_text(prompt, 200, use_turboquant=use_tq)

    print(f'--- {label} ({tps:.0f} tok/s) ---')
    # Show just the generated part (after prompt)
    generated = text[len(prompt):]
    print(generated[:300])
    print()

## 5. Speed & Compression Benchmarks

Benchmark TurboQuant compression, decompression, and attention computation.

In [ ]:
from turboquant import TurboQuantKVCompressor

configs = [
    (2, 6, 256, 64, 'Shakespeare (this model)'),
    (2, 12, 512, 64, 'GPT-2 Small'),
    (1, 12, 1024, 64, 'GPT-2 Small (long ctx)'),
]

bench_device = device
print(f'Benchmarking on: {bench_device}')
print()

all_results = []

for B, H, T, D, name in configs:
    print(f'{name} (B={B}, H={H}, T={T}, D={D})')

    k = torch.randn(B, H, T, D, device=bench_device)
    v = torch.randn(B, H, T, D, device=bench_device)
    q = torch.randn(B, H, T, D, device=bench_device)

    for bits in [3, 4]:
        comp = TurboQuantKVCompressor(D, bits=bits, device=bench_device)

        # Warmup
        ck = comp.compress_keys(k)
        cv = comp.compress_values(v)
        _ = comp.attention_scores(q, ck)
        _ = comp.decompress_values(cv)
        if bench_device == 'cuda':
            torch.cuda.synchronize()

        n_iter = 20

        # TurboQuant path
        t0 = time.time()
        for _ in range(n_iter):
            ck = comp.compress_keys(k)
            cv = comp.compress_values(v)
            scores = comp.attention_scores(q, ck)
            v_hat = comp.decompress_values(cv)
        if bench_device == 'cuda':
            torch.cuda.synchronize()
        t_tq = (time.time() - t0) / n_iter * 1000

        # Standard path
        t0 = time.time()
        for _ in range(n_iter):
            s = torch.matmul(q, k.transpose(-2, -1)) * (1.0 / math.sqrt(D))
            _ = torch.matmul(torch.softmax(s, dim=-1), v)
        if bench_device == 'cuda':
            torch.cuda.synchronize()
        t_std = (time.time() - t0) / n_iter * 1000

        info = comp.memory_savings_info(T, H)
        print(f'  {bits}-bit: TQ={t_tq:.1f}ms, Std={t_std:.1f}ms, '
              f'Compression={info["compression_ratio"]:.1f}x')

        all_results.append({
            'name': name, 'bits': bits,
            't_tq': t_tq, 't_std': t_std,
            'ratio': info['compression_ratio']
        })
    print()

## 6. How TurboQuant Works (Step by Step)

Walk through the compression of a single vector to understand each stage.

In [ ]:
from turboquant import generate_rotation_matrix, generate_qjl_matrix, get_codebook

D = 64
bits = 3

# Create a key vector
torch.manual_seed(123)
k = torch.randn(D)
k_norm = torch.norm(k)
k_unit = k / k_norm

print(f'Original key vector: norm={k_norm:.4f}, first 8 coords: {k[:8].tolist()}')

# STAGE 1: Random rotation
Pi = generate_rotation_matrix(D, seed=42)
k_rotated = k_unit @ Pi.T
print(f'\nAfter rotation: std={k_rotated.std():.4f} (expected: {1/math.sqrt(D):.4f})')

# STAGE 1: Lloyd-Max quantization
centroids, boundaries = get_codebook(D, bits - 1)  # bits-1 for MSE, 1 for QJL
diffs = k_rotated.unsqueeze(-1) - centroids
indices = diffs.abs().argmin(dim=-1)
k_quantized_rot = centroids[indices]
k_mse = (k_quantized_rot @ Pi) * k_norm  # back to original space

mse = ((k - k_mse) ** 2).mean().item()
print(f'After Lloyd-Max {bits-1}-bit quantization: MSE={mse:.6f}')

# STAGE 2: QJL residual correction
residual = k - k_mse
r_norm = torch.norm(residual)
S = generate_qjl_matrix(D, seed=43)
projected = residual @ S.T
signs = torch.sign(projected)
signs[signs == 0] = 1.0

print(f'Residual norm: {r_norm:.4f}')
print(f'QJL signs: {int(signs.sum().item())} positive out of {D}')

# Verify inner product estimation
torch.manual_seed(456)
q = torch.randn(D)

true_ip = (q * k).sum().item()
mse_ip = (q * k_mse).sum().item()

# QJL correction term
q_proj = q @ S.T
qjl_ip = (q_proj * signs).sum().item()
correction = r_norm.item() * math.sqrt(math.pi / 2) / D * qjl_ip
tq_ip = mse_ip + correction

print(f'\nInner product comparison:')
print(f'  True:           {true_ip:.4f}')
print(f'  MSE-only:       {mse_ip:.4f} (error: {abs(true_ip - mse_ip):.4f})')
print(f'  TurboQuant:     {tq_ip:.4f} (error: {abs(true_ip - tq_ip):.4f})')

# Storage comparison
fp16_bits = D * 16
tq_bits = D * (bits - 1) + D * 1 + 16  # MSE indices + QJL signs + norm
print(f'\nStorage: {fp16_bits} bits (FP16) -> {tq_bits} bits (TQ) = {fp16_bits/tq_bits:.1f}x compression')

## Summary

| Metric | 2-bit | 3-bit | 4-bit |
|--------|-------|-------|-------|
| Compression | ~6.5x | ~4.9x | ~3.7x |
| Attn Cosine Sim | ~0.85 | ~0.92 | ~0.97 |
| Quality Impact | Noticeable | Near-lossless | Imperceptible |

**Key takeaways:**
- **3-bit is the sweet spot**: ~5x compression with minimal quality loss
- **Training-free**: No calibration data needed, works on any model
- **Inference-only**: Training uses full precision, TurboQuant applied at inference
- **Memory savings enable**: Longer contexts, larger batch sizes, or running on smaller GPUs